# Lab 1 – In-memory LLM data pipeline (TinyStories + Qwen2.5)

Modified version of the course's `Lab1.ipynb`.

| | Original | This version |
|---|---|---|
| Dataset | WikiText-2 | **TinyStories** (validation split, 21,990 short stories) |
| Tokenizer | GPT-2 (~50k vocab) | **Qwen2.5** (~151k vocab) |
| Document boundaries | texts joined with no separator | **`<|endoftext|>` appended after every story** |
| Concatenation | `sum(lists, [])` | `itertools.chain` (linear time) |

In [1]:
# !pip install -r requirements.txt

In [2]:
from itertools import chain

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

In [3]:
# 1. Load the dataset fully into memory
# The validation split (one parquet file) is small enough to hold in RAM.
dataset = load_dataset("roneneldan/TinyStories", split="validation")
print(f"Number of stories in dataset: {len(dataset)}")
print(dataset[0]["text"][:300])

Number of stories in dataset: 21990
Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They


In [4]:
# 2. Initialize the Qwen2.5 tokenizer
# Qwen already defines <|endoftext|> as both pad and eos, so no pad_token fix is needed (unlike GPT-2).
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
print(f"Vocab size: {len(tokenizer)}")
print(f"pad token: {tokenizer.pad_token!r} ({tokenizer.pad_token_id}) | eos token: {tokenizer.eos_token!r} ({tokenizer.eos_token_id})")

sample = "Once upon a time, there was a little cat."
ids = tokenizer(sample)["input_ids"]
print(ids)
print(tokenizer.convert_ids_to_tokens(ids))

Vocab size: 151665
pad token: '<|endoftext|>' (151643) | eos token: '<|endoftext|>' (151643)
[12522, 5193, 264, 882, 11, 1052, 572, 264, 2632, 8251, 13]
['Once', 'Ġupon', 'Ġa', 'Ġtime', ',', 'Ġthere', 'Ġwas', 'Ġa', 'Ġlittle', 'Ġcat', '.']


In [5]:
# 3. Tokenize with batched .map(), appending <|endoftext|> to every story
# so the model can tell where one story ends and the next begins once we concatenate.
def tokenize_function(examples):
    out = tokenizer(examples["text"], add_special_tokens=False)
    out["input_ids"] = [ids + [tokenizer.eos_token_id] for ids in out["input_ids"]]
    out["attention_mask"] = [mask + [1] for mask in out["attention_mask"]]
    return out

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print(tokenized_ds[0]["input_ids"][:20])  # first 20 token IDs of the first story
lengths = [len(ids) for ids in tokenized_ds["input_ids"]]
print(f"Total tokens: {sum(lengths):,} | mean tokens/story: {sum(lengths) / len(lengths):.1f} | max: {max(lengths)}")

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

[47049, 13, 23297, 5485, 279, 41199, 1803, 323, 1053, 11, 330, 35881, 11, 66181, 11, 697, 1803, 374, 773, 9906]
Total tokens: 4,563,985 | mean tokens/story: 207.5 | max: 997


In [6]:
# 4. Concatenate all stories and slice into fixed-length training blocks
block_size = 128

def group_texts(examples):
    concatenated_inputs = list(chain.from_iterable(examples["input_ids"]))
    concatenated_masks = list(chain.from_iterable(examples["attention_mask"]))

    total_len = (len(concatenated_inputs) // block_size) * block_size
    result_input_ids = [concatenated_inputs[i:i + block_size] for i in range(0, total_len, block_size)]
    result_masks = [concatenated_masks[i:i + block_size] for i in range(0, total_len, block_size)]
    return {"input_ids": result_input_ids, "attention_mask": result_masks}

lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=1000)
print(f"LM training sequences: {len(lm_ds)}")

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

LM training sequences: 35646


In [7]:
# Peek inside one block: story boundaries show up as <|endoftext|>
print(tokenizer.decode(lm_ds[0]["input_ids"]))

Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.<|endoftext|>Once upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She


In [8]:
# 5. DataLoader. Every block is already block_size long, so we just stack.
# For causal LM, labels = input_ids; the model shifts them by one internally.
def collate_fn(batch):
    input_ids = torch.tensor([example["input_ids"] for example in batch], dtype=torch.long)
    attention_mask = torch.tensor([example["attention_mask"] for example in batch], dtype=torch.long)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": input_ids.clone()}

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)

In [9]:
# 6. Iterate through a batch to check shapes
for batch in train_loader:
    print(batch["input_ids"].shape, batch["attention_mask"].shape, batch["labels"].shape)
    break

torch.Size([8, 128]) torch.Size([8, 128]) torch.Size([8, 128])
